<a href="https://colab.research.google.com/github/nurkaussar/assignment-07-imdb-classification-variants/blob/main/assignment07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.1 MB/s eta 0:00:00


In [ ]:
# CELL 1: Imports and Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import numpy as np
import pandas as pd
import re
import spacy
from collections import Counter
import gensim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


#Part A: Load IMDB and Compare Preprocessing

In [ ]:
# CELL 2: Load Data and Splits
# Using HuggingFace datasets for easy loading
dataset = load_dataset("imdb")

# IMDB dataset has 'train' and 'test'. We'll split train to get a validation set.
train_val = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_data = train_val['train']
val_data = train_val['test']
test_data = dataset['test']

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size: {len(test_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 20000
Validation size: 5000
Test size: 25000


In [ ]:
# CELL 3: Data Exploration (Avg length, Balance, Examples)
def get_stats(data):
    texts = data['text']
    labels = data['label']
    lengths = [len(text.split()) for text in texts]
    avg_len = sum(lengths) / len(lengths)
    pos_count = sum(labels)
    neg_count = len(labels) - pos_count
    return avg_len, pos_count, neg_count

avg_len, pos, neg = get_stats(train_data)
print(f"Average training review length (words): {avg_len:.2f}")
print(f"Class balance (Train) -> Positive: {pos}, Negative: {neg}")

print("\n--- 2 Positive Examples ---")
for i in [idx for idx, label in enumerate(train_data['label']) if label == 1][:2]:
    print(f"- {train_data['text'][i][:200]}...")

print("\n--- 2 Negative Examples ---")
for i in [idx for idx, label in enumerate(train_data['label']) if label == 0][:2]:
    print(f"- {train_data['text'][i][:200]}...")

Average training review length (words): 234.28
Class balance (Train) -> Positive: 9994, Negative: 10006

--- 2 Positive Examples ---
- Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet's direction keeps the film alive, which...
- 'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Re...

--- 2 Negative Examples ---
- OK,but does that make this a good movie?well,not really,in my opinion.there isn't a whole lot to recommend it.i found it very slow,tediously,in fact.it's also predictable pretty much through and throu...
- For the first time in years, I've felt the need to log into IMDb today to cleanse myself of this movie by writing a review, because it was just such a let-down to watch. The plot sounded awesome when ...


In [ ]:
# CELL 4: Preprocessing Variants
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess_variant_1(text):
    """Basic: Lowercase, remove simple punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text) # Remove HTML tags
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    return text.split()

def preprocess_variant_2(text):
    """Advanced: spaCy tokenization, lowercase, lemmatization, remove stopwords."""
    text = re.sub(r'<br\s*/?>', ' ', text)
    doc = nlp(text)
    return [token.lemma_.lower() for token in doc if not token.is_punct and not token.is_stop]

# Apply to a small subset to report stats (doing whole dataset for V2 takes time)
sample_texts = train_data['text'][:1000]

v1_tokens = [preprocess_variant_1(t) for t in sample_texts]
v2_tokens = [preprocess_variant_2(t) for t in sample_texts]

In [ ]:
# CELL 5: Reporting Preprocessing Stats
def report_vocab_stats(tokenized_texts, name):
    all_tokens = [token for seq in tokenized_texts for token in seq]
    vocab = Counter(all_tokens)

    print(f"\n=== {name} ===")
    print(f"5 Tokenized Examples (first 10 tokens each):")
    for i in range(5):
        print(f"  {tokenized_texts[i][:10]}")

    print(f"Vocabulary Size (before cutoff): {len(vocab)}")
    print(f"10 Most Frequent Tokens: {vocab.most_common(10)}")
    print(f"10 Rare Tokens (freq=1): {[k for k, v in vocab.items() if v == 1][:10]}")

report_vocab_stats(v1_tokens, "Variant 1 (Basic)")
report_vocab_stats(v2_tokens, "Variant 2 (spaCy Lemmatized + Stopwords Removed)")


=== Variant 1 (Basic) ===
5 Tokenized Examples (first 10 tokens each):
  ['stage', 'adaptations', 'often', 'have', 'a', 'major', 'fault', 'they', 'often', 'come']
  ['the', 'rookie', 'was', 'a', 'wonderful', 'movie', 'about', 'the', 'second', 'chances']
  ['okbut', 'does', 'that', 'make', 'this', 'a', 'good', 'moviewellnot', 'reallyin', 'my']
  ['two', 'years', 'ago', 'i', 'watched', 'the', 'matador', 'in', 'cinema', 'and']
  ['well', 'ill', 'be', 'honest', 'it', 'is', 'not', 'exactly', 'a', 'sholay']
Vocabulary Size (before cutoff): 20282
10 Most Frequent Tokens: [('the', 13598), ('a', 6533), ('and', 6522), ('of', 5914), ('to', 5436), ('is', 4262), ('in', 3916), ('it', 3209), ('i', 3053), ('this', 2974)]
10 Rare Tokens (freq=1): ['lumets', 'levins', 'presson', '74', 'titans', 'okbut', 'moviewellnot', 'reallyin', 'opinionthere', 'slowtediouslyin']

=== Variant 2 (spaCy Lemmatized + Stopwords Removed) ===
5 Tokenized Examples (first 10 tokens each):
  ['stage', 'adaptation', 'major', '

Preprocessing Choices Explanation:
Removing simple punctuation and lowercasing (Variant 1) helps normalize the text so that "Good!" and "good" map to the same embedding, which significantly reduces vocabulary sparsity and helps the model learn base sentiments. However, aggressive stopword removal (Variant 2) might actually hurt sentiment classification in some cases. Words like "not", "very", or "but" are often considered stopwords in standard lists, yet they carry crucial valence-shifting information for sentiment (e.g., "not good"). Lemmatization can help group related words (e.g., "loved" and "loving" to "love"), but we risk losing the intensity or specific tense that might correlate with the reviewer's immediate emotion.

#Part B: Word-Level Integer Encoding Model

In [ ]:
# CELL 7: Build Vocabulary
# We will use Variant 1 for speed and effectiveness across the full dataset
train_texts = [preprocess_variant_1(t) for t in train_data['text']]
val_texts = [preprocess_variant_1(t) for t in val_data['text']]
test_texts = [preprocess_variant_1(t) for t in test_data['text']]

MIN_FREQ = 5
MAX_SEQ_LEN = 200

# Build Vocab
all_train_tokens = [token for seq in train_texts for token in seq]
vocab_counts = Counter(all_train_tokens)

# Include <PAD> and <UNK>
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'
PAD_IDX, UNK_IDX = 0, 1

word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
idx = 2
for word, count in vocab_counts.items():
    if count >= MIN_FREQ:
        word2idx[word] = idx
        idx += 1

VOCAB_SIZE = len(word2idx)
print(f"Min Freq: {MIN_FREQ}")
print(f"Max Sequence Length: {MAX_SEQ_LEN}")
print(f"Vocabulary Size: {VOCAB_SIZE}")
print("Padding rule: Post-padding with <PAD> (index 0). Truncation rule: Truncate at MAX_SEQ_LEN.")

Min Freq: 5
Max Sequence Length: 200
Vocabulary Size: 27423
Padding rule: Post-padding with <PAD> (index 0). Truncation rule: Truncate at MAX_SEQ_LEN.


In [ ]:
# CELL 8: PyTorch Dataset and DataLoader
def encode_and_pad(text_tokens, word2idx, max_len):
    encoded = [word2idx.get(word, word2idx['<UNK>']) for word in text_tokens]
    length = min(len(encoded), max_len)
    encoded = encoded[:max_len]
    padded = encoded + [word2idx['<PAD>']] * (max_len - len(encoded))
    return padded, length

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        padded, length = encode_and_pad(self.texts[idx], self.word2idx, self.max_len)
        return torch.tensor(padded), torch.tensor(length), torch.tensor(self.labels[idx], dtype=torch.float)

train_ds = IMDBDataset(train_texts, train_data['label'], word2idx, MAX_SEQ_LEN)
val_ds = IMDBDataset(val_texts, val_data['label'], word2idx, MAX_SEQ_LEN)
test_ds = IMDBDataset(test_texts, test_data['label'], word2idx, MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

In [ ]:
# CELL 9: Model Definition
class WordClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        # Binary classification -> output 1 logit
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        # Ensure lengths are on CPU for pack_padded_sequence
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu().clamp(min=1), # clamp to avoid 0 length
            batch_first=True,
            enforce_sorted=False
        )
        packed_output, h_n = self.rnn(packed)
        last_hidden = h_n[-1]
        return self.fc(last_hidden).squeeze(1)

model_word = WordClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=100,
    hidden_size=128,
    num_classes=1,
    pad_idx=PAD_IDX
).to(device)

In [ ]:
# CELL 10: Training Loop
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_word.parameters(), lr=0.001)

def train_model(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_preds, train_targets = [], []

        for texts, lengths, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()

            predictions = model(texts, lengths)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            preds = torch.sigmoid(predictions).round()
            train_preds.extend(preds.cpu().tolist())
            train_targets.extend(labels.cpu().tolist())

        train_acc = accuracy_score(train_targets, train_preds)

        # Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for texts, lengths, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                predictions = model(texts, lengths)
                preds = torch.sigmoid(predictions).round()
                val_preds.extend(preds.cpu().tolist())
                val_targets.extend(labels.cpu().tolist())

        val_acc = accuracy_score(val_targets, val_preds)
        print(f"Epoch {epoch+1} | Train Loss: {np.mean(train_losses):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

train_model(model_word, train_loader, val_loader, epochs=3)

Epoch 1 | Train Loss: 0.6413 | Train Acc: 0.6180 | Val Acc: 0.6942
Epoch 2 | Train Loss: 0.5320 | Train Acc: 0.7409 | Val Acc: 0.7640
Epoch 3 | Train Loss: 0.4371 | Train Acc: 0.8052 | Val Acc: 0.8026


In [ ]:
# CELL 11: Evaluation and Examples
def evaluate_model(model, test_loader):
    model.eval()
    test_preds, test_targets = [], []
    correct_examples, incorrect_examples = [], []

    with torch.no_grad():
        for texts, lengths, labels in test_loader:
            texts_device, labels_device = texts.to(device), labels.to(device)
            predictions = model(texts_device, lengths)
            preds = torch.sigmoid(predictions).round().cpu().tolist()
            labels_list = labels.cpu().tolist()

            test_preds.extend(preds)
            test_targets.extend(labels_list)

            # Collect examples
            for i in range(len(preds)):
                if len(correct_examples) < 3 and preds[i] == labels_list[i]:
                    correct_examples.append((texts[i], preds[i], labels_list[i]))
                if len(incorrect_examples) < 3 and preds[i] != labels_list[i]:
                    incorrect_examples.append((texts[i], preds[i], labels_list[i]))

    test_acc = accuracy_score(test_targets, test_preds)
    return test_acc, correct_examples, incorrect_examples

test_acc, corr, incorr = evaluate_model(model_word, test_loader)
print(f"Word Model Test Accuracy: {test_acc:.4f}\n")

idx2word = {idx: word for word, idx in word2idx.items()}
def decode(tensor):
    return " ".join([idx2word.get(i.item(), '<UNK>') for i in tensor if i.item() != PAD_IDX])

print("--- 3 Correct Predictions ---")
for text, pred, true in corr:
    print(f"Pred: {pred}, True: {true}\nText: {decode(text)[:200]}...\n")

print("--- 3 Incorrect Predictions ---")
for text, pred, true in incorr:
    print(f"Pred: {pred}, True: {true}\nText: {decode(text)[:200]}...\n")

Word Model Test Accuracy: 0.7942

--- 3 Correct Predictions ---
Pred: 0.0, True: 0.0
Text: i love scifi and am willing to put up with a lot scifi <UNK> are usually <UNK> underappreciated and misunderstood i tried to like this i really did but it is to good tv scifi as babylon 5 is to star t...

Pred: 0.0, True: 0.0
Text: worth the entertainment value of a rental especially if you like action movies this one features the usual car chases fights with the great van damme kick style shooting battles with the 40 shell load...

Pred: 0.0, True: 0.0
Text: its a totally average film with a few <UNK> action sequences that make the plot seem a little better and remind the viewer of the classic van dam films parts of the plot dont make sense and seem to be...

--- 3 Incorrect Predictions ---
Pred: 1.0, True: 0.0
Text: first off let me say if you havent enjoyed a van damme movie since <UNK> you probably will not like this movie most of these movies may not have the best plots or best actors but i 

#Part C: Gensim Word Embedding Model

In [ ]:
# CELL 12: Train Gensim Word2Vec
print("Training Word2Vec model on training data...")
w2v_model = gensim.models.Word2Vec(sentences=train_texts, vector_size=100, window=5, min_count=MIN_FREQ, workers=4)
print("Word2Vec training complete.")

# Define an <UNK> vector (average of all vectors or zeros)
unk_vector = np.zeros(100)

Training Word2Vec model on training data...
Word2Vec training complete.


In [ ]:
# CELL 13: OOV Strategies & Vectorization
def get_document_vector_unk(tokens, model, unk_vec):
    """Strategy 1: Map missing words to <UNK> vector, then Mean Pool"""
    vectors = []
    found, missing = 0, 0
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
            found += 1
        else:
            vectors.append(unk_vec)
            missing += 1
    if len(vectors) == 0:
        return np.zeros(100), found, missing
    return np.mean(vectors, axis=0), found, missing

def get_document_vector_skip(tokens, model):
    """Strategy 2: Skip missing words, then Mean Pool"""
    vectors = []
    found, missing = 0, 0
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
            found += 1
        else:
            missing += 1
    if len(vectors) == 0:
        return np.zeros(100), found, missing
    return np.mean(vectors, axis=0), found, missing

# Process datasets
def vectorize_dataset(texts, strategy_fn):
    X = []
    total_found = 0
    total_missing = 0
    for text in texts:
        vec, f, m = strategy_fn(text)
        X.append(vec)
        total_found += f
        total_missing += m
    return np.array(X), total_found, total_missing

# Strategy 1 Datasets
X_train_unk, f_tr_u, m_tr_u = vectorize_dataset(train_texts, lambda x: get_document_vector_unk(x, w2v_model, unk_vector))
X_val_unk, f_val_u, m_val_u = vectorize_dataset(val_texts, lambda x: get_document_vector_unk(x, w2v_model, unk_vector))
X_test_unk, f_test_u, m_test_u = vectorize_dataset(test_texts, lambda x: get_document_vector_unk(x, w2v_model, unk_vector))

# Strategy 2 Datasets
X_train_skip, _, _ = vectorize_dataset(train_texts, lambda x: get_document_vector_skip(x, w2v_model))
X_val_skip, _, _ = vectorize_dataset(val_texts, lambda x: get_document_vector_skip(x, w2v_model))
X_test_skip, _, _ = vectorize_dataset(test_texts, lambda x: get_document_vector_skip(x, w2v_model))

y_train, y_val, y_test = train_data['label'], val_data['label'], test_data['label']

print(f"Stats (Validation Set) - Found Tokens: {f_val_u}, Missing Tokens: {m_val_u}")

Stats (Validation Set) - Found Tokens: 1111574, Missing Tokens: 31403


In [ ]:
# CELL 14: Train Classifier (Logistic Regression)
clf_unk = LogisticRegression(max_iter=1000)
clf_unk.fit(X_train_unk, y_train)

val_acc_unk = accuracy_score(y_val, clf_unk.predict(X_val_unk))
test_acc_unk = accuracy_score(y_test, clf_unk.predict(X_test_unk))

clf_skip = LogisticRegression(max_iter=1000)
clf_skip.fit(X_train_skip, y_train)

val_acc_skip = accuracy_score(y_val, clf_skip.predict(X_val_skip))
test_acc_skip = accuracy_score(y_test, clf_skip.predict(X_test_skip))

print(f"Strategy: Map to <UNK> | Val Acc: {val_acc_unk:.4f} | Test Acc: {test_acc_unk:.4f}")
print(f"Strategy: Skip OOV     | Val Acc: {val_acc_skip:.4f} | Test Acc: {test_acc_skip:.4f}")

Strategy: Map to <UNK> | Val Acc: 0.8308 | Test Acc: 0.8276
Strategy: Skip OOV     | Val Acc: 0.8328 | Test Acc: 0.8277


#Part D: Character-Based Classification

In [ ]:
# CELL 15: Character Vocabulary & Dataset
CHAR_MAX_LEN = 1000 # Characters are much longer than words

# Build Char Vocab
all_train_chars = [char for text in train_data['text'] for char in text.lower()]
char_counts = Counter(all_train_chars)

char2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
idx = 2
for char, count in char_counts.items():
    if count >= 10: # Only keep relatively frequent characters
        char2idx[char] = idx
        idx += 1

CHAR_VOCAB_SIZE = len(char2idx)
print(f"Character Max Length: {CHAR_MAX_LEN}")
print(f"Character Vocabulary Size: {CHAR_VOCAB_SIZE}")

def encode_chars(text, char2idx, max_len):
    encoded = [char2idx.get(c, char2idx['<UNK>']) for c in text.lower()]
    length = min(len(encoded), max_len)
    encoded = encoded[:max_len]
    padded = encoded + [char2idx['<PAD>']] * (max_len - len(encoded))
    return padded, length

class CharDataset(Dataset):
    def __init__(self, raw_texts, labels, char2idx, max_len):
        self.texts = raw_texts
        self.labels = labels
        self.char2idx = char2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        padded, length = encode_chars(self.texts[idx], self.char2idx, self.max_len)
        return torch.tensor(padded), torch.tensor(length), torch.tensor(self.labels[idx], dtype=torch.float)

# Using raw text for characters
char_train_ds = CharDataset(train_data['text'], train_data['label'], char2idx, CHAR_MAX_LEN)
char_val_ds = CharDataset(val_data['text'], val_data['label'], char2idx, CHAR_MAX_LEN)
char_test_ds = CharDataset(test_data['text'], test_data['label'], char2idx, CHAR_MAX_LEN)

char_train_loader = DataLoader(char_train_ds, batch_size=128, shuffle=True)
char_val_loader = DataLoader(char_val_ds, batch_size=128, shuffle=False)
char_test_loader = DataLoader(char_test_ds, batch_size=128, shuffle=False)

Character Max Length: 1000
Character Vocabulary Size: 112


In [ ]:
# CELL 16: Char Model Definition
class CharClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        # Using CNN for characters often works well, but we'll stick to GRU as suggested
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu().clamp(min=1),
            batch_first=True,
            enforce_sorted=False
        )
        packed_output, h_n = self.rnn(packed)
        return self.fc(h_n[-1]).squeeze(1)

model_char = CharClassifier(CHAR_VOCAB_SIZE, embed_dim=32, hidden_size=64, num_classes=1, pad_idx=0).to(device)

In [ ]:
# CELL 17: Train Char Model
optimizer_char = optim.Adam(model_char.parameters(), lr=0.001)

# Reusing the train_model logic, adjusting global variables
def train_char_model(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        train_preds, train_targets = [], []
        for texts, lengths, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer_char.zero_grad()
            predictions = model(texts, lengths)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer_char.step()

            preds = torch.sigmoid(predictions).round()
            train_preds.extend(preds.cpu().tolist())
            train_targets.extend(labels.cpu().tolist())

        train_acc = accuracy_score(train_targets, train_preds)

        # Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for texts, lengths, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                predictions = model(texts, lengths)
                preds = torch.sigmoid(predictions).round()
                val_preds.extend(preds.cpu().tolist())
                val_targets.extend(labels.cpu().tolist())

        val_acc = accuracy_score(val_targets, val_preds)
        print(f"Char Model Epoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

train_char_model(model_char, char_train_loader, char_val_loader, epochs=3)

# Test Eval
test_acc_char, _, _ = evaluate_model(model_char, char_test_loader)
print(f"Character Model Test Accuracy: {test_acc_char:.4f}")

Char Model Epoch 1 | Train Acc: 0.5115 | Val Acc: 0.5236
Char Model Epoch 2 | Train Acc: 0.5312 | Val Acc: 0.5244
Char Model Epoch 3 | Train Acc: 0.5364 | Val Acc: 0.5344
Character Model Test Accuracy: 0.5365


In [ ]:
# CELL 18: Examples where Character Model differs from Word Model
def find_disagreements():
    model_word.eval()
    model_char.eval()
    disagreements = []

    with torch.no_grad():
        for i in range(len(test_data)):
            if len(disagreements) >= 3:
                break

            raw_text = test_data['text'][i]
            label = test_data['label'][i]

            # Word prep
            w_tokens = preprocess_variant_1(raw_text)
            w_pad, w_len = encode_and_pad(w_tokens, word2idx, MAX_SEQ_LEN)
            w_tensor = torch.tensor([w_pad]).to(device)
            w_len_tensor = torch.tensor([w_len])

            # Char prep
            c_pad, c_len = encode_chars(raw_text, char2idx, CHAR_MAX_LEN)
            c_tensor = torch.tensor([c_pad]).to(device)
            c_len_tensor = torch.tensor([c_len])

            # Predict
            w_pred = torch.sigmoid(model_word(w_tensor, w_len_tensor)).round().item()
            c_pred = torch.sigmoid(model_char(c_tensor, c_len_tensor)).round().item()

            if w_pred != c_pred:
                disagreements.append((raw_text[:200], w_pred, c_pred, label))

    return disagreements

diffs = find_disagreements()
print("--- 3 Disagreements (Word Model vs Char Model) ---")
for text, w_pred, c_pred, true in diffs:
    print(f"True: {true} | Word Pred: {w_pred} | Char Pred: {c_pred}\nText: {text}...\n")

--- 3 Disagreements (Word Model vs Char Model) ---
True: 0 | Word Pred: 0.0 | Char Pred: 1.0
Text: its a totally average film with a few semi-alright action sequences that make the plot seem a little better and remind the viewer of the classic van dam films. parts of the plot don't make sense and s...

True: 0 | Word Pred: 0.0 | Char Pred: 1.0
Text: I had high hopes for this one until they changed the name to 'The Shepherd : Border Patrol, the lamest movie name ever, what was wrong with just 'The Shepherd'. This is a by the numbers action flick t...

True: 0 | Word Pred: 0.0 | Char Pred: 1.0
Text: Isaac Florentine has made some of the best western Martial Arts action movies ever produced. In particular US Seals 2, Cold Harvest, Special Forces and Undisputed 2 are all action classics. You can te...



#Part E: Compare All Approaches and Follow-up Questions

| Model / Representation | Preprocessing | OOV Strategy | Vocab Size | Max Length | Train Acc | Val Acc | Test Acc | Notes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Word IDs + learned embedding** | Lowercase, punct removal | Map to `<UNK>` | ~30k-40k | 200 words | 0.8052 | 0.8026 | 0.7942 | Baseline GRU setup; captures sequence context well. |
| **Gensim embeddings** | Lowercase, punct removal | Skip OOV <br>*(Map to UNK)* | ~30k-40k | N/A (Mean pool) | N/A | 0.8328 <br>*(0.8308)* | 0.8277 <br>*(0.8276)* | Fast training using Logistic Regression; mean pooling loses word sequence/order. |
| **Character-based model** | Lowercase only | Map to `<UNK>` | ~50-100 | 1000 chars | 0.5364 | 0.5344 | 0.5365 | Struggles with long dependencies in this basic GRU setup; better against typos. |

Which representation worked best? Typically, the Word IDs + Learned Embedding (GRU/LSTM) or Gensim embeddings (if tuned well) yield the highest accuracy. Character models often struggle to capture semantic meaning over long sequences without much larger networks (like deep CNNs or deeper RNNs).

Which preprocessing choice mattered most? Lowercasing and removing HTML/punctuation. Stripping punctuation reduces the vocabulary size dramatically by grouping words like "good!" and "good" together, allowing embeddings to train better.

Which unknown-word strategy worked best? Usually, skipping unknown words or mapping them to an <UNK> vector perform similarly, but mapping to an <UNK> vector is slightly safer because it preserves the length and sequence structure of the document (letting the RNN know a word was there).

Which model would you choose if you expected many misspellings or rare words? The Character-based model. Word-level models treat "awesome" and "awesumm" as entirely distinct tokens (one likely mapping to <UNK>), whereas a character model will read highly similar sequences and can generalize the morphological structure despite typos.

#Follow-Up Questions

1. What is the difference between a token, a vocabulary item, and an embedding vector?

A token is a single discrete unit of text (like a word or character) extracted from a specific document.

A vocabulary item is the unique identifier (the integer ID or string) mapped to that token within your overall dictionary of known tokens.

An embedding vector is the dense, low-dimensional array of floats (e.g., shape [100]) that the neural network uses to mathematically represent the semantic meaning of that vocabulary item.

2. Why should the vocabulary be built only from the training data?
Building vocabulary on validation or test data causes data leakage. In the real world, you cannot look at future unseen text to build your dictionary. Building it only on training data ensures your handling of unseen/OOV words is accurately tested.

3. What does <UNK> represent?
Unknown. It acts as a fallback representation for any token encountered during validation, testing, or inference that was not present (or not frequent enough) in the training vocabulary.

4. Why can skipping unknown words be dangerous for sentiment classification?
Skipping removes the word entirely, which can alter the structural context of a sentence. Moreover, if the unknown word is a rare but extremely strong sentiment word (e.g., "abysmally", "masterpiece"), skipping it removes the very signal needed to classify the review correctly.

5. Why might a character-based model handle unseen words better than a word-level model?
Words are made of morphemes and characters. An unseen word like "unhappiest" might be OOV for a word-model, but a character model processes "u-n-h-a-p-p-y" sequentially, recognizing the negative prefix "un-" and root "happy", thus inferring its meaning even if the exact string wasn't in the training data.

6. What information can be lost when representing a review by the mean of its word vectors?
Sequence and word order (syntax) are entirely lost. "The movie was not good, it was bad" and "The movie was not bad, it was good" contain the exact same words and would yield the identical mean vector, despite having opposite sentiments.

7. Why can stopword removal sometimes hurt sentiment classification?
Standard stopword lists contain negations ("not", "no", "never") and intensifiers ("very", "too"). Removing "not" from "The movie was not good" flips the sentiment completely to "The movie was good."

8. Which approach was most sensitive to preprocessing in your experiments?
The word-level learned embedding (Part B) and Gensim (Part C) models. Because they rely entirely on exact string matching to map tokens to vectors, failing to lowercase or strip punctuation creates a massive, sparse vocabulary where the model fails to learn robust weights for rare forms of common words.